# Step 11 — Score it

*Step 11 of the AI in Industry lab*

---

## Read this before you run anything

You will score the system against ten questions whose correct answers you already know, then change one setting and score it again.

**What you should end up understanding:** How to tell whether a change made things better, with a number instead of an opinion. Almost nobody does this.

| | |
|---|---|
| **Cost** | 30 API calls - takes a few minutes |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes, but see the warning — it is the most expensive notebook in the lab. |

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This notebook makes about 30 API calls and takes several minutes. On a free key that is a large chunk of your quota.</b></div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Run the scoring cell once, read the results properly, and only re-run it after you have deliberately changed something. The comparison between two runs is the whole point — a single score on its own tells you nothing.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Everyone demos on the three questions that work. **The ones who ship, measure.**

This is the step that gets you hired, and almost nobody does it.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()

# Ten questions whose correct answers we already know, verified by hand
# against the PDF. Five categories, two each.
EVAL = [
    ("What is the minimum attendance required in each course?", "75", "lookup"),
    ("How many credits are required for the B.Tech degree?", "160", "lookup"),
    ("I have 68% attendance because of placement drives. What happens?",
     ["condon", "10"], "two-docs"),
    ("I scored 18 out of 60 in formative assessment. What grade do I get?",
     ["R", "21"], "two-docs"),
    ("What are the hostel mess timings?", "REFUSE", "not-in-corpus"),
    ("How much is the tuition fee per semester?", "REFUSE", "not-in-corpus"),
    ("Can I get an exemption if I miss too many classes?", "condon", "paraphrased"),
    ("What do I need to score to not fail the internals?", "21", "paraphrased"),
    ("My attendance is exactly 75%. Am I eligible for the end sem?", "75", "edge"),
    ("My CGPA is exactly 7.0. What class do I get?", "distinction", "edge"),
]

print(f"{len(EVAL)} test questions")

Note row 5 and 6: **the correct answer is a refusal.** Refusing correctly scores a pass.

## The scorer

In [ ]:
def grade(answer, expected):
    low = answer.lower()
    if expected == "REFUSE":
        return "could not find" in low
    needed = expected if isinstance(expected, list) else [expected]
    return all(n.lower() in low for n in needed)


def score(ask, k, label):
    passed = 0
    print(f"\n{label}")
    for question, expected, category in EVAL:
        ok = grade(ask(question, k=k), expected)
        passed += ok
        print(f"  {'PASS' if ok else 'FAIL'}  [{category:13}] {question[:48]}")
    print(f"  --> {passed}/{len(EVAL)}")
    return passed

## Now score three different configurations

This makes 30 API calls and takes a few minutes. It is the most expensive cell in the lab.

In [ ]:
a = score(grounded(tfidf(texts)), 1, "TF-IDF, k=1")
b = score(grounded(tfidf(texts)), 6, "TF-IDF, k=6")
c = score(grounded(embed(texts)), 6, "embeddings, k=6")

print(f"\n  TF-IDF k=1   {a}/10")
print(f"  TF-IDF k=6   {b}/10")
print(f"  embeddings   {c}/10")

## Read the failures, not just the score

Look at **which categories** failed. If they cluster in `paraphrased`, your problem is retrieval, not the model — and changing to a bigger model will not help.

That is a diagnosis with evidence behind it.

## The sentence that gets you hired

> *"I built a RAG chatbot"* — everyone says this.
>
> *"I measured it at 60% on my own eval set, found the failures were retrieval not generation, and got it to 85%"* — almost nobody says this.

You just did the second one, in miniature. **Keep the numbers.**

---

## Now change it yourself

Add your own question. Pick something you can verify in the PDF.

In [ ]:
MY_EVAL = EVAL + [
    # (question, string that must appear in a correct answer, category)
    ("How long do I have to complete the degree?", "seven", "lookup"),
]

ask = grounded(tfidf(texts))
score(ask, 6, "with my extra question")

# Then change ONE thing - k, or the retriever - and score again.
# Write both numbers down. The comparison is the whole point.

---

### Done with step 11

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.